# Neural Networks on GloVe & BERT Embeddings

Trains the two-layer neural-network classifiers from the final paper (Section 5.2.2):
song representations from **GloVe** and fine-tuned **BERT** document embeddings,
concatenated with tag + artist embeddings, predicting the 5 popularity bins.

> Reconstructed from the methodology documented in `reports/final_paper.pdf`.
> Hyperparameters below match the paper exactly: Adam (lr=0.001), 20 epochs,
> batch size 32, validation split 0.2, categorical cross-entropy.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from tensorflow import keras
from tensorflow.keras import layers


## 1. Load data and sample 100k songs (20k per popularity bin)

The paper trains on 100k songs, 20,000 randomly drawn from each of the 5
popularity bins (Section 5).


In [ ]:
df = pd.read_csv('data/song_lyrics_en.csv')
print(df.shape)
print(df['popularity_bin'].value_counts())

# 20k per bin -> 100k total, same sampling as the paper
sampled = (df.groupby('popularity_bin', group_keys=False)
             .apply(lambda g: g.sample(n=20000, random_state=42)))
print(sampled.shape)


## 2. Document embeddings

### 2a. GloVe — mean-pooled `en_core_web_md` word vectors (300-d)

Paper (4.2.3): tokenize each song document (title + lyrics) with spaCy's
`en_core_web_md` and average all token vectors.


In [ ]:
import spacy

nlp = spacy.load('en_core_web_md')

def glove_doc_embedding(text):
    doc = nlp(text)
    vecs = [t.vector for t in doc if t.has_vector]
    return np.mean(vecs, axis=0) if vecs else np.zeros(300)

sampled['song_document'] = sampled['title'].fillna('') + ' ' + sampled['lyrics'].fillna('')
glove_embs = np.stack(sampled['song_document'].apply(glove_doc_embedding))
print('GloVe embeddings:', glove_embs.shape)  # (100000, 300)


### 2b. BERT — fine-tuned `bert-base-uncased`, [CLS] embedding (768-d)

Paper (4.2.3): `bert-base-uncased` fine-tuned on 100k song documents from the
dataset; the document vector is the `[CLS]` embedding.

Fine-tuning (run once; weights reused afterwards):


In [ ]:
from transformers import BertTokenizer, TFBertForSequenceClassification

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# Fine-tune on the 100k song documents with popularity-bin labels, then save:
# model = TFBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=5)
# ... training loop ...
# model.save_pretrained('bert_finetuned_song_docs')


In [ ]:
from transformers import TFBertModel

bert = TFBertModel.from_pretrained('bert_finetuned_song_docs')  # or 'bert-base-uncased'

def bert_doc_embedding(texts, batch_size=32):
    embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        tok = tokenizer(list(batch), padding=True, truncation=True,
                        max_length=512, return_tensors='tf')
        cls_vec = bert(tok)[0][:, 0, :]  # [CLS] token
        embs.append(cls_vec.numpy())
    return np.concatenate(embs, axis=0)

bert_embs = bert_doc_embedding(sampled['song_document'].tolist())
print('BERT embeddings:', bert_embs.shape)  # (100000, 768)


## 3. Tag + artist embeddings, concatenated song representation

Paper (4.2.1–4.2.2): tag embedding (2-d); artist embedding (600-d) averaged with
the mean embedding of featured artists. Song vector = concat(tag, artist, document).
The ablation study (5.1) showed metadata embeddings consistently help, so both
are included here.


In [ ]:
tag_embs = np.load('data/tag_embeddings.npy')      # trained tag embeddings
artist_embs = np.load('data/artist_embeddings.npy')  # trained artist embeddings
# (generate with notebooks/attribute_embeddings.ipynb if missing)

tag_le = LabelEncoder().fit(sampled['tag'])
artist_le = LabelEncoder().fit(sampled['artist'])

tag_vecs = tag_embs[tag_le.transform(sampled['tag'])]
artist_vecs = artist_embs[artist_le.transform(sampled['artist'])]

def song_representation(doc_embs):
    return np.concatenate([tag_vecs, artist_vecs, doc_embs], axis=1)

X_glove = song_representation(glove_embs)
X_bert = song_representation(bert_embs)
print('GloVe song vectors:', X_glove.shape)  # (100000, 902)
print('BERT song vectors:', X_bert.shape)    # (100000, 1370)

# One-hot encoded popularity labels (paper 4.3)
y = keras.utils.to_categorical(
    LabelEncoder().fit_transform(sampled['popularity_bin']), num_classes=5)


## 4. Two-layer neural network (paper 5.2.2)

- 9:1 train/test split · Adam lr=0.001 · categorical cross-entropy
- 20 epochs · batch size 32 · validation split 0.2


In [ ]:
def build_model(input_dim):
    model = keras.Sequential([
        layers.Dense(256, activation='relu', input_shape=(input_dim,)),
        layers.Dense(5, activation='softmax'),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def train_and_evaluate(X, name):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.1, random_state=42)
    model = build_model(X.shape[1])
    model.fit(X_train, y_train, epochs=20, batch_size=32,
              validation_split=0.2, verbose=1)
    pred = model.predict(X_test).argmax(axis=1)
    true = y_test.argmax(axis=1)
    acc = accuracy_score(true, pred)
    f1 = f1_score(true, pred, average='weighted')
    print(f'{name}: accuracy={acc:.4f}, weighted F1={f1:.4f}')
    return acc, f1

results = {}
results['GloVe + NN'] = train_and_evaluate(X_glove, 'GloVe + NN')
results['BERT + NN'] = train_and_evaluate(X_bert, 'BERT + NN')

print()
print('Expected (paper, Table 5):')
print('  GloVe + NN: accuracy=0.30, weighted F1=0.29')
print('  BERT + NN:  accuracy=0.28, weighted F1=0.26')
